# UNSB 리파이너 실험 — 학습 + 480평가 (직접 실행)

제안서 남은 후보 **UNSB(Unpaired Neural Schrodinger Bridge)** 를 plain CUT과 동일 틀(refine_hat)로 비교.
UNSB는 **다단계 생성**(Schrodinger bridge, num_timesteps 스텝) — CUT의 단일 forward와 다름.




## 0. 환경 + 설정 + UNSB 생성함수

In [ ]:
import os, sys, time, glob
import numpy as np
from PIL import Image
import torch
import matplotlib.pyplot as plt
ROOT = os.path.expanduser("~/erang_sr")
UNSB = os.path.join(ROOT, "refiners/unsb_src")
os.chdir(UNSB); sys.path.insert(0, UNSB)          # UNSB 프레임워크 우선
if ROOT not in sys.path: sys.path.append(ROOT)
DEV = "cuda"

# ===== 설정 =====
NUM_TIMESTEPS = 5      # SB 스텝 수 (기본 5)
TAU  = 0.01            # entropy 파라미터
NGF  = 64
N_EPOCHS = 25         # (빨리 보려면 5)
N_DECAY  = 10         # (빨리 보려면 0)
SAVE_EVERY = 5
DR = os.path.join(ROOT, "refiners/data/refine_hat")
EVAL = os.path.join(ROOT, "runs/eval_run")        # 이전 완주런 참조 pngs

def load01(p):
    im = np.asarray(Image.open(p).convert("RGB")).astype(np.float32)/255.0
    return torch.from_numpy(im.transpose(2,0,1)).unsqueeze(0).float()
def save01(t, p):
    a = (t.detach().cpu().clamp(0,1).numpy().transpose(1,2,0)*255).round().astype("uint8")
    Image.fromarray(a).save(p)

def unsb_generate(netG, x01, T=NUM_TIMESTEPS, tau=TAU, ngf=NGF):
    # x01[0,1] → SB 다단계 생성 → [0,1]. sb_model.py test 경로와 동일 공식.
    real_A = (x01*2-1).to(DEV)
    incs = np.array([0]+[1/(i+1) for i in range(T-1)])
    times = np.cumsum(incs); times = times/times[-1]; times = 0.5*times[-1]+0.5*times
    times = np.concatenate([np.zeros(1), times]); times = torch.tensor(times).float().to(DEV)
    bs = real_A.size(0); netG.eval(); Xt_1 = None
    with torch.no_grad():
        for t in range(T):
            if t==0:
                Xt = real_A
            else:
                delta = times[t]-times[t-1]; denom = times[-1]-times[t-1]
                inter = (delta/denom).reshape(-1,1,1,1); scale = (delta*(1-delta/denom)).reshape(-1,1,1,1)
                Xt = (1-inter)*Xt + inter*Xt_1.detach() + (scale*tau).sqrt()*torch.randn_like(Xt)
            tix = (t*torch.ones(bs, device=DEV)).long()
            z = torch.randn(bs, 4*ngf, device=DEV)
            Xt_1 = netG(Xt, tix, z)
    return ((Xt_1+1)/2).clamp(0,1)
print("device:", torch.cuda.get_device_name(0), "| timesteps:", NUM_TIMESTEPS, "| epochs:", N_EPOCHS+N_DECAY)


## 1. 데이터 + 참조 pngs 확인

In [ ]:
for s in ["trainA","trainB","testA","testB"]:
    print(f"  {s}: {len(os.listdir(os.path.join(DR,s)))}장")
print("--- 이전 완주런 참조 ---")
for d in ["sr_hat","cyclegan","plain_cut","hr_ref"]:
    n = len(os.listdir(os.path.join(EVAL,d))) if os.path.isdir(os.path.join(EVAL,d)) else 0
    print(f"  {d}: {n}장", "" if n>=480 else "  <- 없음! cut_res_run_experiment 셀7 먼저 실행 필요")


## 2. UNSB 학습 (다단계 SB, loss 관찰 + 저장)
`--model sb --mode sb --lambda_SB 1.0 --lambda_NCE 1.0`. SB loss가 함께 흐름. UNSB는 데이터로더 2개(zip) 사용.

In [ ]:
sys.argv = ["train.py","--dataroot",DR,"--name","unsb_hat","--mode","sb",
    "--lambda_SB","1.0","--lambda_NCE","1.0","--num_timesteps",str(NUM_TIMESTEPS),"--tau",str(TAU),
    "--display_id","0","--gpu_ids","0","--batch_size","1",
    "--n_epochs",str(N_EPOCHS),"--n_epochs_decay",str(N_DECAY),
    "--load_size","512","--crop_size","256","--print_freq","200"]
from options.train_options import TrainOptions
from data import create_dataset
from models import create_model
opt = TrainOptions().parse(); opt.num_threads = 0
dataset  = create_dataset(opt)
dataset2 = create_dataset(opt)          # UNSB: 두 번째 독립 로더
model = create_model(opt)
total = opt.n_epochs + opt.n_epochs_decay
print(f"\n[unsb_hat] {len(dataset)}장 · {total} epoch (timesteps={NUM_TIMESTEPS})\n")
t0=time.time(); step=0
for epoch in range(opt.epoch_count, total+1):
    for i,(data,data2) in enumerate(zip(dataset,dataset2)):
        if epoch==opt.epoch_count and i==0:
            model.data_dependent_initialize(data,data2); model.setup(opt); model.parallelize()
        model.set_input(data,data2); model.optimize_parameters(); step+=1
        if step%100==0:
            L=model.get_current_losses()
            print(f"  ep{epoch} step{step:>5}  G_GAN {L['G_GAN']:.3f}  NCE {L['NCE']:.3f}  SB {L['SB']:.3f}  ({time.time()-t0:.0f}s)")
    if epoch%SAVE_EVERY==0 or epoch==total:
        model.save_networks("latest"); print(f"  --- ep{epoch} 저장 ({time.time()-t0:.0f}s) ---")
    model.update_learning_rate()
model.save_networks("final")
print(f"[unsb_hat] 완료·저장 → checkpoints/unsb_hat/final_net_G.pth  ({time.time()-t0:.0f}s)")


## 3. UNSB 생성 결과 (테스트 3장, 육안)

In [ ]:
G_unsb = model.netG
tests = sorted(glob.glob(DR+"/testA/*.png"))[:3]
fig, ax = plt.subplots(len(tests),3, figsize=(11,3.6*len(tests))); ax=ax.reshape(len(tests),3)
for r,ta in enumerate(tests):
    x=load01(ta); out=unsb_generate(G_unsb, x)
    imgs=[x[0], out[0].cpu(), load01(ta.replace("testA","testB"))[0]]
    for c,(im,t) in enumerate(zip(imgs,["HAT-SR (input)","UNSB","HR (GT)"])):
        ax[r,c].imshow(im.numpy().transpose(1,2,0)); ax[r,c].axis("off")
        if r==0: ax[r,c].set_title(t)
plt.tight_layout(); plt.show()


## 4. 480장 평가 — UNSB vs CycleGAN vs plain CUT (동일 hr_ref)

In [ ]:
allA = sorted(glob.glob(DR+"/trainA/*.png")) + sorted(glob.glob(DR+"/testA/*.png"))
test_names = set(os.path.basename(p) for p in glob.glob(DR+"/testA/*.png"))
hr_ref = os.path.join(EVAL,"hr_ref")
unsb_dir = os.path.join(EVAL,"unsb"); os.makedirs(unsb_dir, exist_ok=True)
print(f"UNSB {len(allA)}장 생성 중...")
for j,ta in enumerate(allA):
    x=load01(ta); save01(unsb_generate(G_unsb,x)[0], os.path.join(unsb_dir, os.path.basename(ta)))
    if (j+1)%100==0: print(f"  {j+1}/{len(allA)}")
print("생성 완료 · 지표 계산...")
from skimage.metrics import structural_similarity as ssim
import lpips as _L, pyiqa
from pytorch_fid.fid_score import calculate_fid_given_paths
lp=_L.LPIPS(net="alex").to(DEV); niqe=pyiqa.create_metric("niqe",device=DEV)
def psnr(a,b):
    m=float(((a-b)**2).mean()); return 99.0 if m==0 else 10*np.log10(1/m)
conds = {"sr_hat":os.path.join(EVAL,"sr_hat"), "cyclegan":os.path.join(EVAL,"cyclegan"),
         "plain_cut":os.path.join(EVAL,"plain_cut"), "unsb":unsb_dir}
rows={}
for cond,dd in conds.items():
    if not os.path.isdir(dd) or len(os.listdir(dd))<len(allA): print("skip",cond); continue
    ps,ss,lps=[],[],[]
    for name in sorted(test_names):
        out=load01(os.path.join(dd,name)); hr=load01(os.path.join(hr_ref,name))
        a=out[0].numpy().transpose(1,2,0); b=hr[0].numpy().transpose(1,2,0)
        ps.append(psnr(a,b)); ss.append(ssim(a,b,channel_axis=2,data_range=1.0))
        with torch.no_grad(): lps.append(float(lp(out.to(DEV)*2-1,hr.to(DEV)*2-1).mean()))
    nqs=[float(niqe(os.path.join(dd,n))) for n in os.listdir(dd)]
    fid=calculate_fid_given_paths([dd,hr_ref],batch_size=50,device=DEV,dims=2048)
    rows[cond]=(np.mean(ps),np.mean(ss),np.mean(lps),fid,np.mean(nqs))
print(f"\n===== 480장 (HAT 입력, UNSB timesteps={NUM_TIMESTEPS}) =====")
print(f"{'조건':10s} | {'PSNR up':>8} {'SSIM up':>8} {'LPIPS dn':>9} | {'FID dn':>8} {'NIQE dn':>8}")
print("-"*60)
for k in ["sr_hat","cyclegan","plain_cut","unsb"]:
    if k in rows:
        p,s,l,f,n=rows[k]; print(f"{k:10s} | {p:8.3f} {s:8.4f} {l:9.4f} | {f:8.3f} {n:8.3f}")
print("\n판정: UNSB가 plain CUT(FID 171.8)보다 FID 낮고 육안도 좋으면 새 챔피언. 아니면 plain CUT 유지.")


## 5. 육안 최종 비교 (필수) — 지표만 믿지 말 것

In [ ]:
names = sorted(test_names)[:3]
srcs = [("input (HAT-SR)","sr_hat"),("CycleGAN","cyclegan"),("plain CUT","plain_cut"),("UNSB","unsb")]
fig, ax = plt.subplots(3,5, figsize=(18,10.5))
for r,name in enumerate(names):
    cols = [load01(os.path.join(EVAL,d,name))[0] for _,d in srcs] + [load01(os.path.join(hr_ref,name))[0]]
    titles = [t for t,_ in srcs] + ["HR (GT)"]
    for c,(im,t) in enumerate(zip(cols,titles)):
        ax[r,c].imshow(im.numpy().transpose(1,2,0)); ax[r,c].axis("off")
        if r==0: ax[r,c].set_title(t)
plt.tight_layout(); plt.show()
print("UNSB가 (a)입력보다 선명 (b)환각/색이상 없음 (c)plain CUT보다 GT에 가까움 → 채택. 뿌옇거나 색 뜨면 탈락.")
